In [1]:
import numpy as np

import torch

In [2]:
X = np.array([160, 170, 180, 190])
y = np.array([0, 0, 1, 1])

X, y

(array([160, 170, 180, 190]), array([0, 0, 1, 1]))

In [3]:
X_mean = np.mean(X)
X_std = np.std(X)

X_norm = (X - X_mean) / X_std
X_norm

array([-1.34164079, -0.4472136 ,  0.4472136 ,  1.34164079])

In [4]:
X_norm_tensor = torch.tensor(X_norm, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

# 형태 변환 (1,1)형태로
X_norm_tensor = X_norm_tensor.reshape(-1, 1)
y_tensor = y_tensor.reshape(-1, 1)

X_norm_tensor, y_tensor, X_norm_tensor.shape, y_tensor.shape

(tensor([[-1.3416],
         [-0.4472],
         [ 0.4472],
         [ 1.3416]]),
 tensor([[0.],
         [0.],
         [1.],
         [1.]]),
 torch.Size([4, 1]),
 torch.Size([4, 1]))

In [5]:
# torch.nn.Module 상속
class PerceptronModel(torch.nn.Module):
    def __init__(self):
        # 부모 클래스 초기화 실행(반드시 필요)
        super().__init__()

        # 선형 계산
        self.linear = torch.nn.Linear(1, 1)
    
    def forward(self, x):
        # 입력 들어왔을 때 실제 계산 순서 정의
        H = self.linear(x)

        z = torch.sigmoid(H)

        return z

In [6]:
# model 생성
# 랜덤 초기값 고정
torch.manual_seed(42)

model = PerceptronModel()

# model안에 자동으로 만들어진 weight와 bias의 학습 전 초기값을 확인
print('model.linear.weight:', model.linear.weight)
print('model.linear.bias:', model.linear.bias)
print('model 구조:', model)

model.linear.weight: Parameter containing:
tensor([[0.7645]], requires_grad=True)
model.linear.bias: Parameter containing:
tensor([0.8300], requires_grad=True)
model 구조: PerceptronModel(
  (linear): Linear(in_features=1, out_features=1, bias=True)
)


In [7]:
# 학습 전 model 출력 확인
with torch.no_grad():
    z_test = model(X_norm_tensor)

print('z_test shape:', z_test.shape)
print('z_test:\n', z_test)

z_test shape: torch.Size([4, 1])
z_test:
 tensor([[0.4512],
        [0.6197],
        [0.7635],
        [0.8648]])


In [8]:
criterion = torch.nn.BCELoss()
print('criterion 준비 완료:', criterion)

criterion 준비 완료: BCELoss()


In [9]:
learning_rate = 0.1
epochs = 1000

print('learning_rate:', learning_rate)
print('epochs:', epochs)

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

print(list(model.parameters()))

learning_rate: 0.1
epochs: 1000
[Parameter containing:
tensor([[0.7645]], requires_grad=True), Parameter containing:
tensor([0.8300], requires_grad=True)]


In [10]:
for epoch in range(epochs):
    optimizer.zero_grad()

    z = model(X_norm_tensor)
    mean_cost = criterion(z, y_tensor)
    
    mean_cost.backward()
    optimizer.step()

    if epoch % 100 == 0 or epoch == epochs - 1:
        print(
            f'epoch={epoch},'
            f'Cost={mean_cost.item():.6f},'
            f'weight(a)={model.linear.weight.item():.6f},'
            f'bias(b)={model.linear.bias.item():.6f}'
        )
    
    if epoch < 3:
        print(
            f'  (확인용) model.linear.weight.grad={model.linear.weight.grad.item():.6f}'
            f'model.linear.bias.grad={model.linear.bias.grad.item():.6f}'
        )

epoch=0,Cost=0.495464,weight(a)=0.793780,bias(b)=0.812529
  (확인용) model.linear.weight.grad=-0.292415model.linear.bias.grad=0.174793
  (확인용) model.linear.weight.grad=-0.286153model.linear.bias.grad=0.169918
  (확인용) model.linear.weight.grad=-0.280072model.linear.bias.grad=0.165171
epoch=100,Cost=0.178670,weight(a)=2.290082,bias(b)=0.173212
epoch=200,Cost=0.125357,weight(a)=3.002210,bias(b)=0.061586
epoch=300,Cost=0.099283,weight(a)=3.509002,bias(b)=0.026837
epoch=400,Cost=0.082901,weight(a)=3.912263,bias(b)=0.013229
epoch=500,Cost=0.071398,weight(a)=4.250606,bias(b)=0.007116
epoch=600,Cost=0.062789,weight(a)=4.543496,bias(b)=0.004091
epoch=700,Cost=0.056068,weight(a)=4.802371,bias(b)=0.002480
epoch=800,Cost=0.050660,weight(a)=5.034644,bias(b)=0.001570
epoch=900,Cost=0.046207,weight(a)=5.245449,bias(b)=0.001031
epoch=999,Cost=0.042507,weight(a)=5.436657,bias(b)=0.000701


In [11]:
print('학습된 weight(a):', model.linear.weight.item())
print('학습된 bias(b):', model.linear.bias.item())

print('model.linear.weight:', model.linear.weight)
print('model.linear.bias:', model.linear.bias)

print('model.linear.weight.grad:', model.linear.weight.grad)
print('model.linear.bias.grad:', model.linear.bias.grad)


학습된 weight(a): 5.436656951904297
학습된 bias(b): 0.0007013267604634166
model.linear.weight: Parameter containing:
tensor([[5.4367]], requires_grad=True)
model.linear.bias: Parameter containing:
tensor([0.0007], requires_grad=True)
model.linear.weight.grad: tensor([[-0.0185]])
model.linear.bias.grad: tensor([2.6396e-05])


In [12]:
# 입력값 예측
input_height = 185
input_norm = (input_height-X_mean) / X_std

with torch.no_grad():
    input_norm_tensor = torch.tensor([[input_norm]], dtype=torch.float32)
    print('input_norm_tensor shape:', input_norm_tensor.shape)

    z_new = model(input_norm_tensor)

    pred = 1 if z_new.item() >= 0.5 else 0

print(f'키가 {input_height}cm인 사람의 농구선수 확률(z): {z_new.item():.4f}')
if pred == 1:
    print('판별 결과: 농구선수입니다.')
else:
    print('판별 결과: 농구선수가 아닙니다.')

input_norm_tensor shape: torch.Size([1, 1])
키가 185cm인 사람의 농구선수 확률(z): 0.9923
판별 결과: 농구선수입니다.
